In [1]:
import pdfplumber
import csv
import re

In [2]:
column_headers = ['cip_year', 'project_type', 'source_page', 'department','project_name','project_id','start_year','end_year', 
                      'previous_appropriations', 'project_total']
years = {}
cip_year='2025'

In [3]:
cleaned = []
headers = []
first_page = True

with pdfplumber.open(r"C:\Users\vince\Documents\GitHub\CIPBD\Dallas\PDF\\" + f"{cip_year}.pdf") as pdf:
    source_page = 0
    for pg in pdf.pages:

        pg_text = pg.extract_text() or ''
        pg_table = pg.extract_table()
        
        source_page += 1

        if pg_table and "District" in pg_text and "Service" in pg_text and "Completion" in pg_text:
            if first_page:
                headers = pg_table[0]
                first_page = False
            for row in pg_table[1:]:
                
                cleaned_row = [cell.replace('\n', ' ').strip() if cell else '' for cell in row]
                if any('grand total' in str(cell).lower() for cell in cleaned_row):
                    continue
                cleaned_row.append(source_page)
                cleaned.append(cleaned_row)

print(cleaned)
print(headers)

[['Airfield Pavement Evaluation - W214', 'Pavement Maintenance/ Construction', 'Aviation Capital Program', 'Citywide', 'Ongoing', '1,521,648', '1,098,130', '423,518', '0', '0', '0', '1,521,648', 18], ['Airport Emergency Operations Center - W685', 'Terminal', 'Aviation Capital Program', 'Citywide', 'Ongoing', '2,000,000', '0', '2,000,000', '1,300,000', '0', '0', '3,300,000', 18], ['Airport Planning and Advisory Services - 1725', 'Capital Improvement Program', 'Aviation Capital Program', 'Citywide', 'Ongoing', '17,905,000', '12,210,735', '5,694,265', '0', '1,500,000', '4,500,000', '23,905,000', 18], ['Architectural Engineering Roster - W286', 'Capital Improvement Program', 'Aviation Capital Program', 'Citywide', 'Ongoing', '3,154,500', '2,113', '3,152,387', '0', '0', '1,050,000', '4,204,500', 18], ['Aviation Parking Garage - 8738', 'Terminal', 'Aviation Capital Program', 'Citywide', 'Ongoing', '10,755,009', '10,961,056', '(206,048)', '0', '0', '0', '10,755,009', 18], ['Aviation Project R

In [4]:
for i in [7, 8, 9]:
    m = re.search(r'(\d{4})-(\d{2})', headers[i])
    if m:
        yr = "20" + m.group(2)
        years[headers[i]] = yr
        column_headers.append(yr)
    else: 
        years[headers[i]] = cip_year
        column_headers.append(cip_year)
print(years)
print(column_headers)

{'Spent or\nCommitted': '2025', 'FY 2025-26\nBudget': '2026', 'FY 2026-27\nPlanned': '2027'}
['cip_year', 'project_type', 'source_page', 'department', 'project_name', 'project_id', 'start_year', 'end_year', 'previous_appropriations', 'project_total', '2025', '2026', '2027']


In [5]:
# add id, start_year, end_year, and cip_year to rows
ids_raw = {}
ids_clean = []

for row in cleaned:
    
    project_id = ''
    start_year = ''
    end_year = ''
    
    raw_id = row[0][-4:]
    if raw_id in ids_raw: # if id already exists, increment subcount by 1
        ids_raw[raw_id] += 1
    else:
        ids_raw[raw_id] = 1 # if not, set subcount to 1

    project_id = f"{raw_id}.{ids_raw[raw_id]}"

    ids_clean.append(project_id)

    year_cells = []
    for i, cell in enumerate(row):
        if i >= 7 and i < 10:
            year_cells.append((years[headers[i]], cell))

    start_year = next((y for y, cell in year_cells if cell != '0'), '')
    end_year   = next((y for y, cell in reversed(year_cells) if cell != '0'), '')

    for cell in row[:-5]:
        cell = re.sub(r'[\s,()]', '', cell or '')
    print(row)
    
    row.append(project_id)
    row.append(start_year)
    row.append(end_year)
    row.append(cip_year) # cip_year
    
    print(row)

['Airfield Pavement Evaluation - W214', 'Pavement Maintenance/ Construction', 'Aviation Capital Program', 'Citywide', 'Ongoing', '1,521,648', '1,098,130', '423,518', '0', '0', '0', '1,521,648', 18]
['Airfield Pavement Evaluation - W214', 'Pavement Maintenance/ Construction', 'Aviation Capital Program', 'Citywide', 'Ongoing', '1,521,648', '1,098,130', '423,518', '0', '0', '0', '1,521,648', 18, 'W214.1', '2025', '2025', '2025']
['Airport Emergency Operations Center - W685', 'Terminal', 'Aviation Capital Program', 'Citywide', 'Ongoing', '2,000,000', '0', '2,000,000', '1,300,000', '0', '0', '3,300,000', 18]
['Airport Emergency Operations Center - W685', 'Terminal', 'Aviation Capital Program', 'Citywide', 'Ongoing', '2,000,000', '0', '2,000,000', '1,300,000', '0', '0', '3,300,000', 18, 'W685.1', '2025', '2026', '2025']
['Airport Planning and Advisory Services - 1725', 'Capital Improvement Program', 'Aviation Capital Program', 'Citywide', 'Ongoing', '17,905,000', '12,210,735', '5,694,265', '

In [ ]:
# up to this point, cleaned rows are in arrangement of 
# project, service, funding source, council district, completion date
# budget, previous_appropriations, y1, y2, y3, future costs, projec_total, source_page, 
# project_id, start year, end year, cip_year

# new arrangement:
# cip_year, project_type, source_page, service, project_name
# project_id, start_year, end_year, previous_appropriations
# project_total, y1, y2, y3, ... everything else

final = []

def clean_num(cell):
    return re.sub(r'[\s,()]', '', cell or '')    

for row in cleaned:
    numeric_indices = {8, 9, 10, 11, 12}  # positions in new_row: previous_appropriations, project_total, y1, y2, y3
    new_order = [16, 1, 12, 2, 0, 13, 14, 15, 6, 11, 7, 8, 9, 3, 4, 5, 10]
    new_row = [row[i] for i in new_order][:-4]
    #new_row = [clean_num(cell) if i in numeric_indices else cell 
    #            for i, cell in enumerate(new_row)]
    final.append(new_row)
    print(new_row)

['2025', 'Pavement Maintenance/ Construction', 18, 'Aviation Capital Program', 'Airfield Pavement Evaluation - W214', 'W214.1', '2025', '2025', '1098130', '1521648', '423518', '0', '0']
['2025', 'Terminal', 18, 'Aviation Capital Program', 'Airport Emergency Operations Center - W685', 'W685.1', '2025', '2026', '0', '3300000', '2000000', '1,300,000', '0']
['2025', 'Capital Improvement Program', 18, 'Aviation Capital Program', 'Airport Planning and Advisory Services - 1725', '1725.1', '2025', '2027', '12210735', '23905000', '5694265', '0', '1,500,000']
['2025', 'Capital Improvement Program', 18, 'Aviation Capital Program', 'Architectural Engineering Roster - W286', 'W286.1', '2025', '2025', '2113', '4204500', '3152387', '0', '0']
['2025', 'Terminal', 18, 'Aviation Capital Program', 'Aviation Parking Garage - 8738', '8738.1', '2025', '2025', '10961056', '10755009', '206048', '0', '0']
['2025', 'Capital Improvement Program', 18, 'Aviation Capital Program', 'Aviation Project Reserve - W131',

In [ ]:
with open("outputs/2025.csv", "a", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(column_headers)
    writer.writerows(final)


In [ ]:
def clean_num(cell):
    return cell.replace(",","").replace(" ","")

print(clean_num("7 50,000"))

750000
